In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# 1. Load Data
print("Loading data...")
df = pd.read_parquet("streamflow_training_site_3_full.parquet")

# 2. Time-Based Train/Test Split (80/20)
df = df.sort_values("prediction_time")
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# 3. Create X and y
drop_cols = ["target_streamflow", "prediction_time", "site_id"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["target_streamflow"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["target_streamflow"]

# 4. Train XGBoost Baseline
print("Training XGBoost Baseline...")
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)

# 5. Evaluate
y_pred = xgb_model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"XGBoost Baseline Results:")
print(f"RMSE: {rmse:.4f} cfs")
print(f"MAE:  {mae:.4f} cfs")

Loading data...
Training XGBoost Baseline...
XGBoost Baseline Results:
RMSE: 9.0845 cfs
MAE:  4.2314 cfs


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

# --- Configuration ---
LOOKBACK_HOURS = 168
BASE_FEATURES = [
    "snow_depth", "water_temp", "oxygen_dissolved", "streamflow",
    "air_temp", "precipitation", "year", "month", "day", "hour", "season"
]
NUM_FEATURES = len(BASE_FEATURES)

# 1. Load and Split Data
print("Loading data...")
df = pd.read_parquet("streamflow_training_site_3_full.parquet")
df = df.sort_values("prediction_time")

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# Columns are extracted in the exact sequential order for the LSTM
ordered_cols = []
for offset in range(LOOKBACK_HOURS, 0, -1):
    for f in BASE_FEATURES:
        ordered_cols.append(f"{f}_t-{offset}")

X_train_raw = train_df[ordered_cols].values
y_train_raw = train_df["target_streamflow"].values.reshape(-1, 1)

X_test_raw = test_df[ordered_cols].values
y_test_raw = test_df["target_streamflow"].values.reshape(-1, 1)

# 2. Scale the Data
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train_raw)
X_test_scaled = x_scaler.transform(X_test_raw)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train_raw)
y_test_scaled = y_scaler.transform(y_test_raw)

# 3. Reshape for LSTM
X_train_3d = X_train_scaled.reshape(-1, LOOKBACK_HOURS, NUM_FEATURES)
X_test_3d = X_test_scaled.reshape(-1, LOOKBACK_HOURS, NUM_FEATURES)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train_3d, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_3d, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

# 4. LSTM Architecture
class StreamflowLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)
        # Take the output from the final timestep
        final_timestep_out = lstm_out[:, -1, :] 
        return self.fc(final_timestep_out)

model = StreamflowLSTM(input_size=NUM_FEATURES, hidden_size=64, num_layers=2)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 5. Training Loop
epochs = 15
print("Training LSTM...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

# 6. Evaluation
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t).numpy()

# Inverse transform to get actual cubic feet per second (cfs) values
y_pred_actual = y_scaler.inverse_transform(y_pred_scaled)
y_test_actual = y_scaler.inverse_transform(y_test_t.numpy())

rmse = root_mean_squared_error(y_test_actual, y_pred_actual)
mae = mean_absolute_error(y_test_actual, y_pred_actual)

print(f"\nLSTM Results:")
print(f"RMSE: {rmse:.4f} cfs")
print(f"MAE:  {mae:.4f} cfs")

Loading data...
Training LSTM...
Epoch 1/15 - Loss: 0.0171
Epoch 2/15 - Loss: 0.0041
Epoch 3/15 - Loss: 0.0040
Epoch 4/15 - Loss: 0.0035
Epoch 5/15 - Loss: 0.0035
Epoch 6/15 - Loss: 0.0033
Epoch 7/15 - Loss: 0.0034
Epoch 8/15 - Loss: 0.0033
Epoch 9/15 - Loss: 0.0031
Epoch 10/15 - Loss: 0.0030
Epoch 11/15 - Loss: 0.0029
Epoch 12/15 - Loss: 0.0030
Epoch 13/15 - Loss: 0.0028
Epoch 14/15 - Loss: 0.0028
Epoch 15/15 - Loss: 0.0027

LSTM Results:
RMSE: 11.4677 cfs
MAE:  7.6913 cfs


In [3]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

# The persistence baseline assumes the prediction is exactly equal to the streamflow 1 hour ago (streamflow_t-1)
y_pred_persistence = test_df["streamflow_t-1"]
y_test_actual = test_df["target_streamflow"]

persistence_rmse = root_mean_squared_error(y_test_actual, y_pred_persistence)
persistence_mae = mean_absolute_error(y_test_actual, y_pred_persistence)

print(f"Persistence Baseline Results:")
print(f"RMSE: {persistence_rmse:.4f} cfs")
print(f"MAE:  {persistence_mae:.4f} cfs")

Persistence Baseline Results:
RMSE: 9.3169 cfs
MAE:  3.6097 cfs
